# Local-SDFT: Fine-tune a 230M model on your machine

Self-Distillation Fine-Tuning (**SDFT**) is on-policy self-distillation from demos: show the model a few gold pairs, let it answer, then LoRA-train on those self-generated targets — instead of plain **SFT** on off-policy gold text. (Shenfeld et al. study this for continual learning / less forgetting; here we run the same loop on a laptop-sized model.) **GRPO** is the on-policy RL baseline: sample groups of completions, score them, update.

Paper: [Shenfeld et al., 2026 — *Self-Distillation Enables Continual Learning*](https://self-distillation.github.io/SDFT) ([arXiv](https://arxiv.org/abs/2601.19897)).

Base model: [Liquid AI LFM2.5-230M](https://huggingface.co/LiquidAI/LFM2.5-230M) — small enough for Colab T4, Apple Silicon MPS, or even CPU smoke runs.

**One pool, in-sample metrics (no train/test split):** train and score on the same **AlpacaEval 2.0** instructions (`tatsu-lab/alpaca_eval`, full set = **805**). After each arm, report mean `instruction_reward` / refusal / length on those training prompts — online-learning style, like watching `/data` adapt on the conversation stream. Same pragmatism as BFCL / OpenClaw “use the full bank” runs: leverage the whole dataset instead of burning hours on a second held-out eval pass. **Caveat:** this is train-set average, not held-out generalization.


## Setup notes

- **Colab**: Runtime → GPU (T4 is fine). Clone the repo in the next cell, or upload the project zip and `cd` into it. The install cell uninstalls Colab's stale `torchao` so peft LoRA does not hit a version gate.
- **Local**: Apple Silicon (MPS) or CUDA preferred; CPU works for tiny smoke tests.
- Expect the working directory to be the **repo root** (`Local-SDFT/`) so `configs/compare/` and `sdft/` resolve.
- **Smoke vs full** — toggle `SMOKE` in the data cell (single in-sample pool; no held-out split):
  - `SMOKE = True` (default): **16** AE2 instructions — train + train-set average score, GRPO `--max-steps 4` — free Colab finishes quickly.
  - `SMOKE = False`: **all 805** AlpacaEval 2.0 instructions — train + score the same 805 (in-sample). GRPO `--max-steps 16`. One generation pass per arm after training; no second multi-hour held-out eval. Official AE2 **LC win-rate** needs an OpenAI key (optional; not used here).
- CLI cousin (held-out alpaca-cleaned slice, not this notebook’s design): `python scripts/run_batch1_comparison.py --num-train 32 --num-eval 16 --max-grpo-steps 16`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Install loose pins (Colab / fresh venv).
# peft>=0.19 raises ImportError if torchao is installed but <0.16 (Colab often
# ships 0.10). We don't use torchao quantization for this 230M LoRA path — peft
# falls through to the default nn.Linear dispatcher when torchao is absent.
%pip install -q "torch>=2.6" "transformers>=4.54" "trl>=0.19" "peft>=0.15" "datasets>=2.19" "accelerate>=0.33" "pyyaml>=6.0"
%pip uninstall -y torchao

REPO_URL = "https://github.com/lin826/Local-SDFT.git"
REPO_DIR = Path("Local-SDFT")

cwd = Path.cwd().resolve()
if (cwd / "sdft").is_dir() and (cwd / "configs" / "compare").is_dir():
    ROOT = cwd
    print(f"using existing repo root: {ROOT}")
elif (cwd / REPO_DIR / "sdft").is_dir():
    ROOT = (cwd / REPO_DIR).resolve()
    print(f"using cloned repo: {ROOT}")
else:
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL])
    ROOT = (cwd / REPO_DIR).resolve()
    print(f"cloned into: {ROOT}")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["PYTHONPATH"] = str(ROOT)

print("cwd:", Path.cwd())
print("PYTHONPATH:", os.environ["PYTHONPATH"])

## Three methods (same train budget; in-sample AE2 score)

```
 tatsu-lab/alpaca_eval  (single pool — SMOKE: 16 / full: 805)
      |
      +---> gold targets (davinci-003 `output`) --> LoRA SFT  (gold SFT)
      |
      +---> few-shot teacher gen ---> self targets -> LoRA SFT  (SDFT)
      |         ^ demos sampled from the same AE2 pool
      |
      +---> prompts only -----------> GRPO rollouts + reward  (GRPO)
      |
      +---> after train: one gen pass on the SAME prompts
               -> mean instruction_reward / refusal / length
                 (train-set average — online-learning style)
```

- **Gold SFT** — AE2 ships instruction + davinci-003 `output`; we use that `output` as the gold `response` target (proxy gold for SFT, not human alpaca-cleaned labels).
- **SDFT** — model rewrites each answer (few-shot demos from the same pool), then SFT on those rewrites.
- **GRPO** — sample completion groups, score with a local instruction reward, optimize relatively.
- **Score** — regenerate on the training prompts; report train-set averages (not held-out generalization). Same full-bank idea as BFCL / OpenClaw and the `/data` online loop.

In [ ]:
import json
from pathlib import Path

from sdft.alpacaeval_ablation import ALPACA_EVAL_FULL_N, load_alpaca_eval_examples
from sdft.grpo_train import examples_to_grpo_jsonl

# --- toggle ---
SMOKE = True  # True: 16 in-sample. False: full AE2 805 train + same-pool score.
# --------------

if SMOKE:
    NUM_TRAIN = 16
    MAX_GRPO_STEPS = 4
else:
    NUM_TRAIN = ALPACA_EVAL_FULL_N  # all 805 AlpacaEval 2.0 instructions
    MAX_GRPO_STEPS = 16

TRAIN_DATASET = "tatsu-lab/alpaca_eval"  # via alpaca_eval.json (805 rows)

# Single pool: AE2 instructions for train AND later in-sample scoring.
# Gold targets: davinci-003 `output` from the dataset (mapped to `response`).
train_ex = load_alpaca_eval_examples(num_examples=NUM_TRAIN)
assert len(train_ex) == NUM_TRAIN, f"expected {NUM_TRAIN} AE2 rows, got {len(train_ex)}"
missing_gold = sum(1 for e in train_ex if not (e.get("response") or "").strip())
assert missing_gold == 0, (
    f"{missing_gold}/{NUM_TRAIN} AE2 rows lack davinci-003 output — cannot build gold SFT"
)

data_dir = Path("data/compare")
data_dir.mkdir(parents=True, exist_ok=True)
gold_path = data_dir / "batch1_gold.jsonl"
grpo_path = data_dir / "batch1_grpo.jsonl"

with gold_path.open("w", encoding="utf-8") as fh:
    for e in train_ex:
        fh.write(
            json.dumps(
                {
                    "prompt": e["prompt"],
                    "response": e["response"],  # davinci-003 proxy gold
                    "sdft_response": e["response"],
                },
                ensure_ascii=False,
            )
            + "\n"
        )

examples_to_grpo_jsonl(train_ex, grpo_path)
print(
    f"SMOKE={SMOKE}  n={NUM_TRAIN}/{ALPACA_EVAL_FULL_N} ({TRAIN_DATASET})  "
    f"in-sample score on same pool  max_grpo_steps={MAX_GRPO_STEPS}"
)
print(f"wrote {gold_path} ({NUM_TRAIN} rows; gold=davinci-003 AE2 output)")
print(f"wrote {grpo_path} ({NUM_TRAIN} rows)")
print("sample prompt:", train_ex[0]["prompt"][:100].replace("\n", " "), "…")
print("sample gold:  ", train_ex[0]["response"][:100].replace("\n", " "), "…")



## SDFT teacher pass

For each **training** prompt (AE2 pool), show the model a few gold demos (`num_shots=2` sampled from the same in-sample pool) and let it answer. Those self-generated strings become the SDFT targets.

If you hit CUDA/MPS OOM, drop `max_new_tokens` (e.g. 96) or `generation.batch_size` to 1 in the config / cell below.


In [ ]:
import json
from pathlib import Path

from sdft.config import load_config
from sdft.generate import generate_responses
from sdft.utils import pick_device

sdft_path = Path("data/compare/batch1_sdft.jsonl")
sdft_cfg = load_config("configs/compare/batch1_sdft.yaml")
sdft_cfg.data.num_examples = NUM_TRAIN
sdft_cfg.generation.num_shots = 2
sdft_cfg.generation.out_path = str(sdft_path)
# Safer defaults for Colab T4 / MPS smoke
sdft_cfg.generation.batch_size = 1
sdft_cfg.generation.max_new_tokens = 128

device = pick_device()
print(f"device={device}  teacher on {NUM_TRAIN} examples, num_shots=2")

try:
    gens = generate_responses(sdft_cfg, train_ex, device)
except RuntimeError as err:
    if "out of memory" in str(err).lower() or "oom" in str(err).lower():
        raise RuntimeError(
            "OOM during teacher generate. Reduce generation.max_new_tokens "
            "(try 64–96) and/or set generation.batch_size=1, then re-run this cell."
        ) from err
    raise

rows = []
for ex, gen in zip(train_ex, gens):
    if not gen:
        continue
    rows.append({"prompt": ex["prompt"], "response": ex["response"], "sdft_response": gen})

sdft_path.parent.mkdir(parents=True, exist_ok=True)
with sdft_path.open("w", encoding="utf-8") as fh:
    for row in rows:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"wrote {len(rows)}/{NUM_TRAIN} SDFT pairs -> {sdft_path}")
if rows:
    print("sample sdft_response:", rows[0]["sdft_response"][:200].replace("\n", " "), "…")

## Train three adapters (`batch_size=1` style)

Configs under `configs/compare/` already use LoRA `r=8`, `max_length=512`, and `batch_size=1` for SFT/SDFT (same vibe as the online-learning demo).

**GRPO** steps come from `MAX_GRPO_STEPS` (4 when `SMOKE`, else 16). Full uncapped GRPO: omit `--max-steps` — much slower on Colab.

In [ ]:
import subprocess
import sys
from pathlib import Path


def run_mod(*args: str) -> None:
    cmd = [sys.executable, "-m", *args]
    print("+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd, cwd=str(Path.cwd()))


gold_path = Path("data/compare/batch1_gold.jsonl")
sdft_path = Path("data/compare/batch1_sdft.jsonl")
grpo_path = Path("data/compare/batch1_grpo.jsonl")

out_sft = Path("outputs/compare/batch1-sft-gold")
out_sdft = Path("outputs/compare/batch1-sdft")
out_grpo = Path("outputs/compare/batch1-grpo")

run_mod(
    "sdft.train",
    "--config", "configs/compare/batch1_sft_gold.yaml",
    "--data", str(gold_path),
    "--target", "gold",
    "--output-dir", str(out_sft),
)

run_mod(
    "sdft.train",
    "--config", "configs/compare/batch1_sdft.yaml",
    "--data", str(sdft_path),
    "--target", "sdft",
    "--output-dir", str(out_sdft),
)

# SMOKE: 4 steps. Full 805 in-sample path: 16. Omit --max-steps for uncapped GRPO (slow).
print(f"GRPO max_steps={MAX_GRPO_STEPS} (SMOKE={SMOKE})", flush=True)
run_mod(
    "sdft.grpo_train",
    "--config", "configs/compare/batch1_grpo.yaml",
    "--data", str(grpo_path),
    "--output-dir", str(out_grpo),
    "--max-steps", str(MAX_GRPO_STEPS),
)

print("adapters:", out_sft, out_sdft, out_grpo)


## Score on the training pool (in-sample / online-learning style)

After training, run **one** generation pass over the **same** `NUM_TRAIN` prompts and report mean `instruction_reward`, refusal rate, and mean chars for **base / sft_gold / sdft / grpo**. Missing adapters are skipped.

This is a **train-set average** (like watching online learning on the conversation stream) — **not** held-out generalization and **not** official AlpacaEval LC win-rate (that needs `OPENAI_API_KEY` + GPT-4 judge). Davinci-003 refs are used only for lexical overlap in the heuristic reward.


In [ ]:
import re
from pathlib import Path

import torch
from peft import PeftModel

from sdft.config import ModelConfig, load_config
from sdft.peft_utils import adapter_ready
from sdft.rewards import instruction_reward
from sdft.utils import load_model, load_tokenizer, pick_device

REFUSAL_RE = re.compile(r"(?i)\b(i('m| am) sorry|i can('t|not) (assist|help)|as an ai)\b")

MODEL_NAME = load_config("configs/compare/batch1_sdft.yaml").model.name  # LiquidAI/LFM2.5-230M
# In-sample: score the same prompts we trained on (no held-out split).
prompts = [e["prompt"] for e in train_ex]
golds = [e["response"] for e in train_ex]  # davinci-003 refs (heuristic overlap only)

arms = {
    "base": None,
    "sft_gold": Path("outputs/compare/batch1-sft-gold"),
    "sdft": Path("outputs/compare/batch1-sdft"),
    "grpo": Path("outputs/compare/batch1-grpo"),
}

print(
    f"in-sample score n={len(prompts)} AE2 train prompts (SMOKE={SMOKE}) "
    f"— train-set average, not held-out",
    flush=True,
)


@torch.inference_mode()
def generate_eval(adapter_dir: Path | None, max_new_tokens: int = 128) -> list[str]:
    device = pick_device()
    cfg = ModelConfig(name=MODEL_NAME)
    tokenizer = load_tokenizer(cfg)
    tokenizer.padding_side = "left"
    base = load_model(cfg, device)
    model = (
        PeftModel.from_pretrained(base, str(adapter_dir))
        if adapter_dir is not None and adapter_ready(adapter_dir)
        else base
    )
    model.eval()
    outs: list[str] = []
    for i, prompt in enumerate(prompts):
        if (i + 1) % 50 == 0 or i == 0:
            print(f"  gen {i + 1}/{len(prompts)}…", flush=True)
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        enc = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(device)
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        new = out[:, enc["input_ids"].shape[1] :]
        outs.append(tokenizer.decode(new[0], skip_special_tokens=True).strip())
    del model, base
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return outs


# Per-arm generations + rewards kept for the qualitative sample cell below.
arm_results: dict[str, dict] = {}
rows = []
for name, adapter in arms.items():
    if adapter is not None and not adapter_ready(adapter):
        print(f"skip {name}: adapter missing at {adapter}")
        continue
    print(f"scoring {name}…", flush=True)
    gens = generate_eval(adapter)
    rewards = instruction_reward(gens, gold=golds)
    refuse_flags = [bool(REFUSAL_RE.search(g)) for g in gens]
    chars = [len(g) for g in gens]
    n = max(len(gens), 1)
    arm_results[name] = {
        "gens": gens,
        "rewards": rewards,
        "refusals": refuse_flags,
        "adapter": adapter,
    }
    rows.append(
        {
            "arm": name,
            "n": n,
            "mean_reward": round(sum(rewards) / n, 3),
            "refusal_rate": round(sum(refuse_flags) / n, 3),
            "mean_chars": round(sum(chars) / n, 1),
        }
    )

print()
print("| arm | n | mean_reward | refusal_rate | mean_chars |")
print("|-----|---|-------------|--------------|------------|")
for r in rows:
    print(
        f"| {r['arm']} | {r['n']} | {r['mean_reward']:.3f} | "
        f"{r['refusal_rate']:.3f} | {r['mean_chars']:.1f} |"
    )
print()
print("metrics = train-set average (in-sample / online-learning style)")



## Qualitative sample — same prompt, all arms

Mean reward tables can hide *how* arms differ: a small loss gap or aggregate score often looks flat while one prompt shows **base refusing**, gold SFT hedging, and **SDFT actually answering**. This cell picks one **in-sample** scored prompt (preferring an SDFT win on `instruction_reward` / non-refusal) and regenerates or reuses the same-prompt outputs for **base / sft_gold / sdft / grpo** side by side.

Why it matters: loss curves and mean rewards are easy to misread on a 230M smoke run; a shared-prompt qualitative check is the fastest way to see whether SDFT’s on-policy targets changed behavior (e.g. helpful how-tos from the README `/perf` examples) rather than only fitting tokens.


In [ ]:
# Qualitative: same prompt across arms — prefer an SDFT local-score win.
import html
import re
from pathlib import Path

import torch
from IPython.display import HTML, display
from peft import PeftModel

from sdft.config import ModelConfig, load_config
from sdft.peft_utils import adapter_ready
from sdft.rewards import instruction_reward
from sdft.utils import load_model, load_tokenizer, pick_device

REFUSAL_RE = re.compile(r"(?i)\b(i('m| am) sorry|i can('t|not) (assist|help)|as an ai)\b")
TRUNCATE_AT = 600  # chars shown in the table; full length noted when truncated

# README /perf qualitative prompts (practical how-tos where base often refuses).
FALLBACK_PROMPTS = [
    "Sew a button on a shirt",
    "How do I make apple juice?",
]

ARM_ORDER = ["base", "sft_gold", "sdft", "grpo"]
ARM_PATHS = {
    "base": None,
    "sft_gold": Path("outputs/compare/batch1-sft-gold"),
    "sdft": Path("outputs/compare/batch1-sdft"),
    "grpo": Path("outputs/compare/batch1-grpo"),
}


def _ensure_arm_results() -> dict[str, dict]:
    """Reuse score-cell `arm_results` if present; otherwise regenerate from adapters."""
    existing = globals().get("arm_results")
    if isinstance(existing, dict) and "sdft" in existing and "base" in existing:
        print(f"reusing arm_results from score cell ({len(existing)} arms, "
              f"n={len(existing['sdft']['gens'])})")
        return existing

    prompts_local = list(globals().get("prompts") or [e["prompt"] for e in train_ex])
    golds_local = list(globals().get("golds") or [e["response"] for e in train_ex])
    model_name = globals().get("MODEL_NAME") or load_config(
        "configs/compare/batch1_sdft.yaml"
    ).model.name
    gen_fn = globals().get("generate_eval")

    print("arm_results missing/incomplete — regenerating per-arm gens…", flush=True)

    @torch.inference_mode()
    def _gen(adapter_dir: Path | None, max_new_tokens: int = 128) -> list[str]:
        if callable(gen_fn) and adapter_dir in (
            None,
            ARM_PATHS["sft_gold"],
            ARM_PATHS["sdft"],
            ARM_PATHS["grpo"],
        ):
            # Prefer the score cell's generate_eval (same greedy settings).
            return gen_fn(adapter_dir, max_new_tokens=max_new_tokens)
        device = pick_device()
        cfg = ModelConfig(name=model_name)
        tokenizer = load_tokenizer(cfg)
        tokenizer.padding_side = "left"
        base = load_model(cfg, device)
        model = (
            PeftModel.from_pretrained(base, str(adapter_dir))
            if adapter_dir is not None and adapter_ready(adapter_dir)
            else base
        )
        model.eval()
        outs: list[str] = []
        for prompt in prompts_local:
            messages = [{"role": "user", "content": prompt}]
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            enc = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(device)
            out = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
            new = out[:, enc["input_ids"].shape[1] :]
            outs.append(tokenizer.decode(new[0], skip_special_tokens=True).strip())
        del model, base
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return outs

    rebuilt: dict[str, dict] = {}
    for name, adapter in ARM_PATHS.items():
        if adapter is not None and not adapter_ready(adapter):
            print(f"skip {name}: adapter missing at {adapter}")
            continue
        gens = _gen(adapter)
        rewards = instruction_reward(gens, gold=golds_local)
        rebuilt[name] = {
            "gens": gens,
            "rewards": rewards,
            "refusals": [bool(REFUSAL_RE.search(g)) for g in gens],
            "adapter": adapter,
        }
    globals()["arm_results"] = rebuilt
    return rebuilt


def _score_row(i: int, results: dict[str, dict]) -> dict[str, float | bool]:
    out: dict[str, float | bool] = {}
    for arm in ARM_ORDER:
        if arm not in results:
            continue
        out[f"{arm}_reward"] = float(results[arm]["rewards"][i])
        out[f"{arm}_refuse"] = bool(results[arm]["refusals"][i])
    return out


def _sdft_advantage(scores: dict[str, float | bool]) -> float | None:
    if "sdft_reward" not in scores or "base_reward" not in scores:
        return None
    return float(scores["sdft_reward"]) - float(scores["base_reward"])


def pick_sdft_win_index(results: dict[str, dict], n: int) -> tuple[int | None, str]:
    """Prefer SDFT strictly better than base; ideally also > gold SFT and GRPO."""
    candidates: list[tuple[tuple, int]] = []
    for i in range(n):
        s = _score_row(i, results)
        if "sdft_reward" not in s or "base_reward" not in s:
            continue
        sdft_r = float(s["sdft_reward"])
        base_r = float(s["base_reward"])
        if sdft_r <= base_r:
            continue
        beat_gold = ("sft_gold_reward" not in s) or (sdft_r > float(s["sft_gold_reward"]))
        beat_grpo = ("grpo_reward" not in s) or (sdft_r > float(s["grpo_reward"]))
        # Sort key: prefer beating all arms, then larger margin vs base, then non-refusal.
        key = (
            int(beat_gold and beat_grpo),
            int(beat_gold),
            int(beat_grpo),
            sdft_r - base_r,
            int(not bool(s.get("sdft_refuse", True))),
            int(bool(s.get("base_refuse", False))),
        )
        candidates.append((key, i))

    if candidates:
        candidates.sort(reverse=True)
        best_key, idx = candidates[0]
        if best_key[0]:
            reason = "SDFT strictly better than base, sft_gold, and grpo"
        elif best_key[1] or best_key[2]:
            reason = "SDFT strictly better than base and at least one other arm"
        else:
            reason = "SDFT strictly better than base (others tied or missing)"
        return idx, reason

    # Fallback 1: best relative SDFT advantage vs base (even if not a strict win).
    best_i, best_adv = None, float("-inf")
    for i in range(n):
        adv = _sdft_advantage(_score_row(i, results))
        if adv is None:
            continue
        if adv > best_adv:
            best_adv, best_i = adv, i
    if best_i is not None:
        return best_i, (
            f"no strict SDFT>base win in n={n}; "
            f"best relative advantage vs base = {best_adv:+.3f}"
        )
    return None, f"no scored rows available (n={n})"


def generate_one(prompt: str, adapter_dir: Path | None, max_new_tokens: int = 128) -> str:
    device = pick_device()
    model_name = globals().get("MODEL_NAME") or load_config(
        "configs/compare/batch1_sdft.yaml"
    ).model.name
    cfg = ModelConfig(name=model_name)
    tokenizer = load_tokenizer(cfg)
    tokenizer.padding_side = "left"
    base = load_model(cfg, device)
    model = (
        PeftModel.from_pretrained(base, str(adapter_dir))
        if adapter_dir is not None and adapter_ready(adapter_dir)
        else base
    )
    model.eval()
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(device)
    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    new = out[:, enc["input_ids"].shape[1] :]
    decoded = tokenizer.decode(new[0], skip_special_tokens=True).strip()
    del model, base
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return decoded


def truncate_display(text: str, limit: int = TRUNCATE_AT) -> tuple[str, str]:
    if len(text) <= limit:
        return text, ""
    note = f" [truncated; full length={len(text)} chars]"
    return text[:limit] + "…", note


results = _ensure_arm_results()
n_scored = len(next(iter(results.values()))["gens"]) if results else 0
prompts_list = list(globals().get("prompts") or [e["prompt"] for e in train_ex])
golds_list = list(globals().get("golds") or [e["response"] for e in train_ex])

idx, pick_reason = pick_sdft_win_index(results, n_scored)
selection_mode = "train_scored"
prompt_used: str
arm_outputs: dict[str, dict]

if idx is not None:
    prompt_used = prompts_list[idx]
    arm_outputs = {}
    for arm in ARM_ORDER:
        if arm not in results:
            continue
        gen = results[arm]["gens"][idx]
        arm_outputs[arm] = {
            "output": gen,
            "reward": float(results[arm]["rewards"][idx]),
            "refusal": bool(results[arm]["refusals"][idx]),
        }
    print(f"selected train-set index={idx}  ({pick_reason})")
else:
    # Fallback 2: README qualitative how-to (regenerate all arms on that prompt).
    selection_mode = "readme_fallback"
    prompt_used = FALLBACK_PROMPTS[0]
    print(
        f"no usable scored SDFT comparison (SMOKE may be tiny). "
        f"Falling back to README qualitative prompt: {prompt_used!r}"
    )
    arm_outputs = {}
    for arm, adapter in ARM_PATHS.items():
        if adapter is not None and not adapter_ready(adapter):
            print(f"skip {arm}: adapter missing")
            continue
        print(f"generating {arm} on fallback prompt…", flush=True)
        gen = generate_one(prompt_used, adapter)
        rew = instruction_reward([gen], gold=None)[0]
        arm_outputs[arm] = {
            "output": gen,
            "reward": float(rew),
            "refusal": bool(REFUSAL_RE.search(gen)),
        }
    pick_reason = (
        "README /perf fallback (Sew a button…) — no SDFT>base win and "
        "no relative advantage row in current scored set"
    )

# --- display ---
print()
print(f"mode={selection_mode}")
print(f"reason={pick_reason}")
print(f"prompt={prompt_used!r}")
print()

header = (
    "<h3>Same-prompt comparison</h3>"
    f"<p><b>Selection:</b> {html.escape(pick_reason)}</p>"
    f"<p><b>Prompt:</b> {html.escape(prompt_used)}</p>"
)
rows_html = [
    "<table style='border-collapse:collapse;width:100%;font-size:13px'>",
    "<tr>"
    "<th style='border:1px solid #ccc;padding:6px;text-align:left'>arm</th>"
    "<th style='border:1px solid #ccc;padding:6px;text-align:left'>reward</th>"
    "<th style='border:1px solid #ccc;padding:6px;text-align:left'>refusal</th>"
    "<th style='border:1px solid #ccc;padding:6px;text-align:left'>output</th>"
    "</tr>",
]
for arm in ARM_ORDER:
    if arm not in arm_outputs:
        continue
    o = arm_outputs[arm]
    shown, note = truncate_display(o["output"])
    refuse_s = "yes" if o["refusal"] else "no"
    rows_html.append(
        "<tr>"
        f"<td style='border:1px solid #ccc;padding:6px;vertical-align:top'><b>{html.escape(arm)}</b></td>"
        f"<td style='border:1px solid #ccc;padding:6px;vertical-align:top'>{o['reward']:.3f}</td>"
        f"<td style='border:1px solid #ccc;padding:6px;vertical-align:top'>{refuse_s}</td>"
        f"<td style='border:1px solid #ccc;padding:6px;vertical-align:top;"
        f"white-space:pre-wrap;font-family:ui-monospace,monospace'>"
        f"{html.escape(shown)}{html.escape(note)}</td>"
        "</tr>"
    )
rows_html.append("</table>")
display(HTML(header + "".join(rows_html)))

# Also print a plain markdown table for Colab / export without HTML.
print("| arm | reward | refusal | output (truncated) |")
print("|-----|--------|---------|-------------------|")
for arm in ARM_ORDER:
    if arm not in arm_outputs:
        continue
    o = arm_outputs[arm]
    shown, note = truncate_display(o["output"], limit=160)
    flat = (shown + note).replace("|", "\\|").replace("\n", " ")
    print(
        f"| {arm} | {o['reward']:.3f} | "
        f"{'yes' if o['refusal'] else 'no'} | {flat} |"
    )


## What to try next

1. **Full AE2 in-sample run** — set `SMOKE = False` (805 train + same-pool score, GRPO 16 steps) and re-run from the data cell. Metrics are train-set averages (online-learning style), not held-out generalization. Smoke first to verify the pipeline.
2. **Custom local JSONL** — drop Alpaca-style rows into `data/my_dataset.jsonl` and point a config at it with `dataset: json` + `data_files: ...` (same in-sample score pattern works).
3. **Online learning web UI** — locally: `python -m web.app`, open `/data` (each message is +/0/− tone feedback on the prior reply, then a tiny LoRA SDFT update).
4. **OpenClaw tool-use** — see `docs/openclaw-rl-eval.md` for the ReTool-style loop and boxed reward (full-bank pragmatism, same spirit as this notebook).
5. **Held-out batch-1 CLI** — `python scripts/run_batch1_comparison.py --num-train 32 --num-eval 16 --max-grpo-steps 16` (alpaca-cleaned train/eval split — different design from this notebook).

Happy tinkering — 230M is small enough that curiosity is the bottleneck, not VRAM.
